### Sposób 4: decyduj się na funkcję pomocnicze zamiast na skomplikowane wyrazenia

Zadanie: zdekodować zapytanie URL.

In [ ]:
from urllib.parse import parse_qs
my_values = parse_qs('red=5&blue=0&green=', keep_blank_values=True)
print(repr(my_values))

print('Red: ', my_values.get('red'))
print('Green: ', my_values.get('green'))
print('Blue: ', my_values.get('blue'))




{'red': ['5'], 'blue': ['0'], 'green': ['']}
Red:  ['5']
Green:  ['']
Blue:  ['0']


Dobrze byloby w momnecie gdy jest pusty parametr przypisywać 0

In [ ]:
red = int(my_values.get("red", [""])[0] or 0) #ale tutaj nam się robi juz balagan
green = my_values.get("green", [""])[0] or 0
blue = int(my_values.get("blue", [""])[0] or 0)

print(f"Red: {red!r}")
print(f"Green: {green!r}")
print(f"Blue: {blue!r}")

Red: 5
Green: 0
Blue: 0


Zdefiniujmy, wiec sobie funkcje

In [9]:
def get_first_element(value,key,default=0):
    found = value.get(key, [""])
    if found[0]:
        return int(found[0])
    return default


red = get_first_element(my_values, "red")
green = get_first_element(my_values, "green")
blue = get_first_element(my_values, "blue")

print(f"Red: {red!r}")
print(f"Green: {green!r}")
print(f"Blue: {blue!r}")

Red: 5
Green: 0
Blue: 0


#### ZAPAMIĘTAĆ:
- Skomplikowane wyrazenia przeniesc do funkcji pomocniczych, zwlaszcza jesli ta sama logika ma byc wieloktornie wykorzystywana

### Sposob 5: Zamiast indeksowania wybieraj rozpakowanie wielu operacji przypisania

Python oferuje typ wbudowany `tuple` pozwalajacy tworzyc niemodyfikowalne uporzadkowane sekwencje wartosci. `Tuple` moze byc pusta badz zawierac pojedynczy element

```python
no_snack = ()
snack = ("chipsy",)
```

In [ ]:
snack_calories = {
    "chipsy": 140,
    "jogurt": 155,
    "jajecznica": 200,
}
#zamiana na krotki
items= tuple(snack_calories.items())
print(items)



(('chipsy', 140), ('jogurt', 155), ('jajecznica', 200))


**Gdzie takie sytuacje (dict → tuple of items) są przydatne?**

1. **Niezmienność** — krotka jest immutable, więc nikt nie zmieni zawartości "w locie". Przydatne, gdy przekazujesz dane dalej i chcesz mieć gwarancję, że nie zostaną zmodyfikowane.

2. **Klucze w słowniku** — tylko obiekty hashable (np. tuple) mogą być kluczami. Np. `(nazwa, kalorie)` jako klucz w innym dict albo w `set()` do sprawdzania unikalnych par.

3. **Stała kolejność przy iteracji** — `tuple(...)` "zamraża" aktualną kolejność par; przy długich pętlach kolejność się nie zmieni nawet gdy gdzie indziej modyfikujesz oryginalny dict.

4. **Sortowanie** — listę krotek `(klucz, wartość)` łatwo sortować po kluczu lub po wartości, np. `sorted(items, key=lambda x: x[1])` żeby posortować po kaloriach.

5. **Serializacja / logowanie** — reprezentacja krotek jest przewidywalna; łatwo zapisać "snapshot" słownika (np. do logu) bez ryzyka, że ktoś zmieni dane przed zapisem.

In [13]:
# Rozpakowywanie tuple

name, calories = items[0]
print(f"Name: {name}, Calories: {calories}")

Name: chipsy, Calories: 140


Lepsza metoda rozakowywania tuples

In [15]:
snacks = [("Maslo", 140), ("Jogurt", 155), ("Jajecznica", 200)]
for i in range(len(snacks)):
    item = snacks[i]
    name = item[0]
    calories = item[1]
    print(f"{name}: {calories}")

print("Lepsza metoda")
for name, calories in snacks:
    print(f"{name}: {calories}")



Maslo: 140
Jogurt: 155
Jajecznica: 200
Lepsza metoda
Maslo: 140
Jogurt: 155
Jajecznica: 200


### Sposób 6: Krotkę jednoelementową zawsze umieszczaj w nawiasie

Mamy tutaj rozne sposoby definiowania krotek

In [1]:
first = (1,2,3)
second = (1,2,3,)
third = 1,2,3
fourth = 1,2,3, 
assert first == second == third == fourth

Mamy jeszcze 3 przypadki tworzenia krotek

In [ ]:
empty = () #pusta krotka
single = (1,) #krotka z jednym elementem - wymaga umieszczenia przecinka
error = (1) #to jest int, a nie tuple
assert empty == single == error




AssertionError: 

Wazne jest zwracac uwage na definiowanie zmiennych, funkcji 

przykladowo:

to_refund = calculate_refund(
    get_oreder(user,order.id)
), <--- ten przecinek sprawi, ze wynik nie bedzie int tylko tuple

### Sposób 7: W prostej logice osadzonej uzywaj wyrazen warunkowych

Koncetruje sie ten sposob w okol wyrazen warunkowych

In [4]:
i = 3 
x = "parzysta" if i % 2 == 0 else "nieparzysta"
print(x)

nieparzysta


Niekoniecznie powyzszy kod jest lepszy od funkcji, w przypadku rozbudowy to wieksze mozliwosci daje nam:

In [ ]:
if i % 2 == 0:
    x = "parzysta"
else:
    x = "nieparzysta"
    
print(x)



nieparzysta


Jezeli naprawde chcemy uzyskac duza zwiezlosc i umiescic te logike w pojedynczym wyrazeniu - to przeniesmy ja do funkcji

taka funkcja moze nyc uzywana wiele razy, a nie jak w przypadku wyrazen warunkowych jednorazowo

In [ ]:
def number_category(i):
    if u%2==0:
        return "parzysta"
    else:
        return "nieparzysta"

x = number_category(i)


Więc kiedy, co wybrac?

- Nalezy unukac wyrazen warunkowych gdy musi ono zostac podzielone na wiele wierszy
`x = (my_long_function_call(1,2,3) if some_condition else my_long_function_call(4,5,6))` - jak widac kod jest nieczytelny 

### Wyrazenia przypisania a wyrazenia warunkowe - polaczenie Sposob 7 z 8 